# 03 — Final Model Evaluation and Research Questions

**QM640 | Locked holdout evaluation | RQ1–RQ4**

Run **after Notebook 01 and Notebook 02**. This notebook loads the frozen validation-selected configurations and refits each on **train + validation**. It evaluates on held-out quarters exactly once; model selection is **not altered** after results are seen.

- **RQ1:** Does the validation-selected primary model beat a historical-median baseline on test MAE?
- **RQ2:** Which variables influence the selected model's predictions? Permutation importance on untouched holdout rows is descriptive and can be unstable with few quarters; it is *not* coefficient significance or causality.
- **RQ3:** Compare all five frozen candidate algorithms using identical test observations; distinguish selection based on validation from observed holdout ranking.
- **RQ4:** Compare reported-only versus enhanced features **within the primary model family** using development-selected hyperparameters.
- **Uncertainty:** Paired company-block bootstrap intervals for differences in absolute error; short-panel temporal dependence and only two held-out quarters limit inference.

Only export computed findings. Never invent model accuracy or p-values.

## 1. Read immutable handoff and frozen model selection

In [ ]:
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from scipy import stats

SEED = 640
SOURCE = Path("statistical_outputs")
if not (SOURCE/"prepared_company_quarters.csv").exists():
    SOURCE = Path("/content/statistical_outputs")
MODEL = SOURCE.parent/"model_outputs"
OUT = SOURCE.parent/"final_evaluation_outputs"
OUT.mkdir(parents=True, exist_ok=True)
assert (SOURCE/"prepared_company_quarters.csv").exists(), "Run Notebook 01 first."
assert (MODEL/"selection_manifest.json").exists(), "Run Notebook 02 first."
prepared = SOURCE/"prepared_company_quarters.csv"
handoff = json.loads((SOURCE/"handoff_manifest.json").read_text())
selection = json.loads((MODEL/"selection_manifest.json").read_text())
sha = hashlib.sha256(prepared.read_bytes()).hexdigest()
assert sha == handoff["prepared_csv_sha256"] == selection["source_sha256"], "Input/selection hash mismatch."
df = pd.read_csv(prepared,low_memory=False)
df = df[df.split.isin(["train","validation","test"])].copy()
assert df.target_growth_pct.notna().all()
dev = df[df.split.isin(["train","validation"])].copy()
test = df[df.split.eq("test")].copy()
assert set(test.target_q.astype(str)) == set(selection["test_target_quarters"])
assert set(dev.target_q.astype(str)).isdisjoint(set(test.target_q.astype(str)))
assert max(dev.target_q.astype(str)) < min(test.target_q.astype(str))
assert len(test)>0
print("Development rows:",len(dev),"Held-out rows:",len(test),"Held-out quarters:",sorted(test.target_q.unique()))
print("Model selection is frozen from Notebook 02.")

## 2. Refit frozen candidates on development data and predict once on test data

In [ ]:
categorical = ["INDEX_INDUSTRY","quarter_of_year"]
feature_sets = selection["feature_sets"]
all_features = sorted(set(sum((list(v) for v in feature_sets.values()),[])))
assert set(all_features).issubset(df.columns)
for c in all_features:
    if c not in categorical:
        dev[c] = pd.to_numeric(dev[c],errors="coerce").replace([np.inf,-np.inf],np.nan)
        test[c] = pd.to_numeric(test[c],errors="coerce").replace([np.inf,-np.inf],np.nan)
for c in categorical:
    dev[c] = dev[c].fillna("Unknown").astype(str)
    test[c] = test[c].fillna("Unknown").astype(str)
def estimator(name,params):
    if name=="Historical median": return DummyRegressor(**params)
    if name=="Linear Regression": return LinearRegression()
    if name=="Ridge": return Ridge(**params)
    if name=="Random Forest": return RandomForestRegressor(random_state=SEED,n_jobs=-1,**params)
    if name=="Gradient Boosting": return GradientBoostingRegressor(random_state=SEED,**params)
    raise ValueError(name)
def build(name,params,features):
    nums=[c for c in features if c not in categorical]
    cats=[c for c in features if c in categorical]
    try: enc=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
    except TypeError: enc=OneHotEncoder(handle_unknown="ignore",sparse=False)
    prep=ColumnTransformer([("numeric",Pipeline([("impute",SimpleImputer(strategy="median")),
         ("scale",StandardScaler())]),nums),("categorical",enc,cats)])
    return Pipeline([("preprocess",prep),("model",estimator(name,params))])
def parse_key(key):
    feature_set,name,params_json = key.split("|",2)
    return feature_set,name,json.loads(params_json)
keys=sorted(set([selection["primary"]["key"]] + list(selection["rq3_keys"].values())
                 + list(selection["rq4_keys"].values())))
models,rows,pred_by_key={},[],{}
for key in keys:
    feature_set,name,params=parse_key(key)
    cols=feature_sets[feature_set]
    pipe=build(name,params,cols)
    pipe.fit(dev[cols],dev.target_growth_pct)
    pred=pipe.predict(test[cols])
    assert len(pred)==len(test) and np.isfinite(pred).all()
    models[key]=pipe
    pred_by_key[key]=pred
    rows.append({"key":key,"feature_set":feature_set,"algorithm":name,
                 "MAE":mean_absolute_error(test.target_growth_pct,pred),
                 "RMSE":np.sqrt(mean_squared_error(test.target_growth_pct,pred)),
                 "R2":r2_score(test.target_growth_pct,pred)})
scores=pd.DataFrame(rows).sort_values("MAE")
scores.to_csv(OUT/"heldout_model_metrics.csv",index=False)
prediction_table=test[["COMPANY_KEY","target_q","target_growth_pct"]].reset_index(drop=True).copy()
for i,key in enumerate(keys):
    prediction_table[f"prediction_{i:02d}"]=pred_by_key[key]
(OUT/"prediction_key_map.json").write_text(json.dumps({f"prediction_{i:02d}":k for i,k in enumerate(keys)},indent=2))
prediction_table.to_csv(OUT/"heldout_predictions.csv",index=False)
display(scores[["feature_set","algorithm","MAE","RMSE","R2"]])
print("Primary preselected model:",selection["primary"]["key"])

## 3. Dependence-aware exploratory paired loss intervals

In [ ]:
def company_bootstrap_delta(pred_a,pred_b,n_boot=2000,seed=640):
    y=test.target_growth_pct.to_numpy(float)
    delta=np.abs(y-np.asarray(pred_a))-np.abs(y-np.asarray(pred_b))
    grouped=pd.DataFrame({"company":test.COMPANY_KEY.to_numpy(),"delta":delta})
    # Bootstrap companies (cluster blocks) and retain all observations belonging to each.
    blocks=[g.delta.to_numpy(float) for _,g in grouped.groupby("company",sort=False)]
    rng=np.random.default_rng(seed)
    values=[]
    for _ in range(n_boot):
        sample=rng.integers(0,len(blocks),size=len(blocks))
        values.append(np.concatenate([blocks[i] for i in sample]).mean())
    low,high=np.quantile(values,[.025,.975])
    return {"MAE_improvement_pp":float(delta.mean()),
            "company_bootstrap_CI_low":float(low),
            "company_bootstrap_CI_high":float(high),
            "n_companies":len(blocks),"n_quarters":test.target_q.nunique(),
            "interpretation":"exploratory; company bootstrap does not address few common time periods"}

## 4. RQ1 — Predictive value against a historical baseline

In [ ]:
primary_key=selection["primary"]["key"]
baseline_key=selection["rq3_keys"]["Historical median"]
rq1=company_bootstrap_delta(pred_by_key[baseline_key],pred_by_key[primary_key],seed=SEED)
rq1.update({"RQ":"RQ1","baseline":"Historical median","selected_model":primary_key})
display(pd.DataFrame([rq1]))
pd.DataFrame([rq1]).to_csv(OUT/"RQ1_baseline_comparison.csv",index=False)
if primary_key==baseline_key:
    print("Primary model IS the historical median baseline; improvement is identically zero.")
elif rq1["company_bootstrap_CI_low"]>0:
    print("Observed test MAE advantage, exploratory company-bootstrap interval excludes zero.")
else:
    print("No robust positive advantage supported by this exploratory interval.")

## 5. RQ2 — Predictive feature importance for the selected model

In [ ]:
selected_features=feature_sets[selection["primary"]["feature_set"]]
primary=models[primary_key]
importance=permutation_importance(primary,test[selected_features],test.target_growth_pct,
    scoring="neg_mean_absolute_error",n_repeats=15,random_state=SEED)
feature_importance=pd.DataFrame({
    "feature":selected_features,
    "MAE_increase_pp":importance.importances_mean,
    "repeat_std":importance.importances_std}).sort_values("MAE_increase_pp",ascending=False)
feature_importance.to_csv(OUT/"RQ2_permutation_importance.csv",index=False)
display(feature_importance)
ax=feature_importance.set_index("feature").MAE_increase_pp.sort_values().plot.barh(figsize=(9,6),
    title="RQ2: change in held-out MAE after permuting each feature")
ax.set_xlabel("Increase in MAE (percentage points; larger means greater reliance)")
plt.tight_layout(); plt.savefig(OUT/"RQ2_permutation_importance.png"); plt.show()
print("Permutation importance is not statistical coefficient significance or causality.")

## 6. RQ3 — Frozen candidate algorithm comparison

In [ ]:
rq3=[]
primary_algo=selection["primary"]["algorithm"]
for algorithm,key in selection["rq3_keys"].items():
    res=company_bootstrap_delta(pred_by_key[baseline_key],pred_by_key[key],seed=SEED)
    res.update({"algorithm":algorithm,"key":key,
                "MAE":float(mean_absolute_error(test.target_growth_pct,pred_by_key[key])),
                "RMSE":float(np.sqrt(mean_squared_error(test.target_growth_pct,pred_by_key[key]))),
                "R2":float(r2_score(test.target_growth_pct,pred_by_key[key]))})
    rq3.append(res)
rq3_table=pd.DataFrame(rq3).sort_values("MAE")
rq3_table.to_csv(OUT/"RQ3_model_comparison.csv",index=False)
display(rq3_table[["algorithm","MAE","RMSE","R2","MAE_improvement_pp","company_bootstrap_CI_low","company_bootstrap_CI_high"]])
print("The lowest held-out MAE is descriptive; model selection was fixed by validation, not changed here.")
print("Held-out ranking:",rq3_table.iloc[0]["algorithm"],"; preselected primary:",primary_algo)

## 7. RQ4 — Fixed-family reported vs engineered features

In [ ]:
key_reported=selection["rq4_keys"]["reported_only"]
key_enhanced=selection["rq4_keys"]["reported_plus_engineered"]
assert parse_key(key_reported)[1]==parse_key(key_enhanced)[1], "RQ4 algorithms must match."
rq4=company_bootstrap_delta(pred_by_key[key_reported],pred_by_key[key_enhanced],seed=SEED)
rq4.update({"RQ":"RQ4","algorithm":parse_key(key_reported)[1],
            "reported_key":key_reported,"enhanced_key":key_enhanced})
pd.DataFrame([rq4]).to_csv(OUT/"RQ4_feature_set_comparison.csv",index=False)
display(pd.DataFrame([rq4]))
if rq4["algorithm"]=="Historical median":
    print("Caution: a constant historical-median model ignores features; RQ4 is not substantively testable with this family.")
print("Positive improvement favors engineered features. This is a held-out paired comparison, not causal evidence.")

## 8. Consolidated RQ1–RQ4 evidence and limitations

In [ ]:
rq_summary=pd.DataFrame([
 {"RQ":"RQ1","question":"Does selected model beat historical median on test MAE?",
  "evidence":f"MAE difference baseline minus selected = {rq1['MAE_improvement_pp']:.4f} pp",
  "qualification":"Exploratory company-block interval; few target quarters."},
 {"RQ":"RQ2","question":"Which variables matter for the frozen selected model?",
  "evidence":"Permutation ranking exported: RQ2_permutation_importance.csv",
  "qualification":"Model-specific, holdout-dependent, noncausal."},
 {"RQ":"RQ3","question":"Which candidate model has the lowest holdout error?",
  "evidence":f"Descriptive lowest heldout MAE: {rq3_table.iloc[0]['algorithm']}",
  "qualification":"Primary selection was made on validation, not chosen by test ranking."},
 {"RQ":"RQ4","question":"Do engineered features improve the chosen model family?",
  "evidence":f"Reported minus enhanced MAE difference = {rq4['MAE_improvement_pp']:.4f} pp",
  "qualification":"Exploratory paired comparison; undefined feature value for constant baseline."}
])
rq_summary.to_csv(OUT/"RQ_final_evidence_summary.csv",index=False)
display(rq_summary)
(OUT/"evaluation_provenance.json").write_text(json.dumps({
 "source_sha256":sha,"primary_key":primary_key,"test_quarters":sorted(test.target_q.astype(str).unique().tolist()),
 "n_test":len(test),"model_selection_source":str(MODEL/"selection_manifest.json"),
 "limits":["Only a small number of held-out quarters","Extreme PAT growth denominators",
           "Source filing availability may not have been checked","Company bootstrap does not handle common time shocks",
           "No tuning performed on held-out outcomes"]},indent=2))
print("Review all exported tables before writing final conclusions:", OUT.resolve())

## Reporting principles

1. Do not substitute the Notebook 01 coefficient tests for predictive feature importance (RQ2).
2. Do not select a different algorithm after comparing held-out errors in Notebook 03.
3. Treat company-block intervals as exploratory because quarters are few; they do not resolve systematic time shocks.
4. Report the original continuous PAT-growth target, missing balance-sheet predictors, and negative/near-zero PAT denominator risk.
5. If the winner is a constant baseline, RQ4 needs a **predeclared nonconstant comparison** to answer feature-value questions; do not label an identical baseline comparison as evidence that features lack value.